# LiteAgents agentic loop demos

Two small agents that demonstrate the complete loop:

```text
user prompt -> model -> tool call -> tool result -> model -> final answer
```

The first agent uses one fixed model. The second uses JEV to select a model before each model turn

In [1]:
%pip install -q "liteagents @ git+https://github.com/BerriAI/liteagents.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
ERROR: Package 'liteagents' requires a different Python: 3.9.6 not in '>=3.10'
Note: you may need to restart the kernel to use updated packages.


## Credentials

The fixed-model demo needs `OPENAI_API_KEY`. The JEV demo also needs `TYPESAFE_API_KEY`

In [ ]:
import os
from getpass import getpass

for name in ("OPENAI_API_KEY", "TYPESAFE_API_KEY"):
    if not os.getenv(name):
        value = getpass(f"{name}, leave blank to skip: " ).strip()
        if value:
            os.environ[name] = value

## Demo 1: support agent with a tool

The model cannot know the order status by itself. It must call `get_order_status`, receive the result, and use that result in its final answer

In [ ]:
from typing import Any

from liteagents import LiteAgentOptions, Tool, query


class GetOrderStatus(Tool):
    name = "get_order_status"
    description = "Look up the current status of an order"
    input_schema = {
        "type": "object",
        "properties": {"order_id": {"type": "string"}},
        "required": ["order_id"],
    }

    async def execute(self, input: dict[str, Any]) -> str:
        orders = {
            "A-100": "Shipped yesterday, expected Friday",
            "B-200": "Waiting for payment confirmation",
        }
        return orders.get(str(input["order_id"]), "Order not found")


support_options = LiteAgentOptions(
    model="openai/gpt-5.4-mini",
    system="Help the customer. Always use the order tool for order status questions",
    tools=[GetOrderStatus()],
    max_turns=4,
)

async for event in query(
    prompt="Where is order A-100?",
    options=support_options,
):
    print(type(event).__name__, event)

Expected loop:

1. The model requests `get_order_status(order_id="A-100")`
2. LiteAgents executes the Python function
3. The tool result is added to the conversation
4. The model answers with the shipping status

## Demo 2: JEV-routed PR review agent

JEV chooses between a fast model and a stronger model. The selected model must inspect the PR through a tool before classifying its risk

In [ ]:
from liteagents import JevModelRouter, JevTier


class GetPullRequest(Tool):
    name = "get_pull_request"
    description = "Return the title, summary, and diff for a pull request"
    input_schema = {
        "type": "object",
        "properties": {"pr_number": {"type": "integer"}},
        "required": ["pr_number"],
    }

    async def execute(self, input: dict[str, Any]) -> str:
        if int(input["pr_number"]) != 42:
            return "Pull request not found"
        return """
Title: Rotate API signing keys
Summary: Adds zero-downtime key rotation
Diff:
+ accept current and previous signing keys during a 10 minute overlap
+ delete the previous key after the overlap
+ add rollback and authentication tests
""".strip()


jev_router = JevModelRouter(
    tiers=(
        JevTier(
            name="FAST",
            model="openai/gpt-5.4-mini",
            description="Routine lookups, summaries, and isolated edits",
        ),
        JevTier(
            name="REASONING",
            model="openai/gpt-5.4",
            description="Security, architecture, migrations, and difficult tradeoffs",
        ),
    ),
    fallback_model="openai/gpt-5.4",
)

review_options = LiteAgentOptions(
    model_router=jev_router,
    system=(
        "Review pull request risk. Use get_pull_request before answering. "
        "Return low, medium, or high risk with a short reason"
    ),
    tools=[GetPullRequest()],
    max_turns=4,
)

async for event in query(
    prompt="Classify the deployment risk of PR 42",
    options=review_options,
):
    print(type(event).__name__, event)

Expected loop:

1. JEV selects a model for the request
2. The selected model requests `get_pull_request(pr_number=42)`
3. LiteAgents executes the tool and returns the PR data
4. JEV can select again before the follow-up model turn
5. The model returns a risk classification